# Gout — kiểm chứng C03–C04
Notebook này **không cần GPU**. Nó kiểm tra history mới và append output thật đã lưu từ C02. C05 chưa được thực hiện.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
print('Uploaded:', zip_name)

In [ ]:
import zipfile, os, pathlib
work = pathlib.Path('/content/gout_c03_c04')
work.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(work)
root = work / 'gout_units_v01'
assert (root / 'unit03_chat_engine.py').exists()
os.chdir(root)
print('Working dir:', os.getcwd())

In [ ]:
!python -m unittest discover -s tests -p "test_*.py" -v

In [ ]:
from unit03_chat_engine import new_history, append_turn
history_a = new_history()
history_b = new_history()
assert history_a == [] and history_b == [] and history_a is not history_b
print('C03 OK:', history_a)

In [ ]:
import json
from pathlib import Path
pred_path = Path('evidence/c02_chat_engine_real_20261004/predictions.jsonl')
pred = json.loads(pred_path.read_text(encoding='utf-8').strip())
history = new_history()
history = append_turn(history, pred['question'], pred.get('answer'), status=pred['status'])
print(json.dumps(history, ensure_ascii=False, indent=2))
assert pred['status'] == 'ok'
assert len(history) == 2
assert history[0] == {'role':'user', 'content':pred['question']}
assert history[1] == {'role':'assistant', 'content':pred['answer']}
print('C04 OK: assistant trong history đúng bằng output model thật của C02')

In [ ]:
bad = append_turn(history, 'Q lỗi', None, status='error')
assert bad == history
print('Lượt lỗi không được append: OK')

In [ ]:
import shutil
src = Path('evidence/c03_c04_from_real_c02_20261004')
archive = shutil.make_archive('/content/Gout_C03_C04_result', 'zip', src)
print(archive)
files.download(archive)